In [ ]:
# INSTALL IF NECESSARY

%pip install pandas numpy pillow matplotlib tensorflow scikit-learn joblib

In [ ]:
# -*- coding: utf-8 -*-

**AER850 Project Phase II: Data Preprocessing**


Mars Surface / Curiosity image set (Kaggle)

In [ ]:
# SETUP
from pathlib import Path
import json  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()  # finds current directory
DATA_DIR = PROJECT_ROOT / "Data" / "Mars_Surface_and_Curiosity_Image"
IMAGE_DIR = DATA_DIR / "images"  # complete image collection location
OUT_DIR = PROJECT_ROOT / "processed"  # output will be saved here

# every image gets resized to IMG_SIZE x IMG_SIZE
IMG_SIZE = 224

BATCH_SIZE = 16  # process a few images at a time; can lower this if memory is limited
SEED = 42   # allows for reproducibility of random operations

# error messages for missing directories
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Missing images folder: {IMAGE_DIR}")
OUT_DIR.mkdir(exist_ok=True)

# check that directories are correct
print("Project:", PROJECT_ROOT)
print("Dataset:", DATA_DIR)
print("Generated files:", OUT_DIR)

Files are split into training, validation, and test based on the .csv's that are included with the dataset.


In [ ]:
# Load the CSV's

split_files = { 
    "train": "Train_CSV.csv",
    "validation": "Validation_CSV.csv",
    "test": "Test_CSV.csv",
}

tables = []  # empty list that will hold three tables
for split, filename in split_files.items():
    table = pd.read_csv(DATA_DIR / filename, dtype={"JPG": "string", "LABELS": "Int64"})
    table = table.rename(columns={"JPG": "file", "LABELS": "label"})
    table["split"] = split  # add the split name to every row in this table
    tables.append(table)  # add this table to the list
df = pd.concat(tables, ignore_index=True)  # stack the tables and number rows from zero
display(df.head(10))  # show the first ten rows
display(df["split"].value_counts())  # count rows in each split

The labels are defined as shown below:

| ID | Target / Feature | ID | Target / Feature | ID | Target / Feature |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | apxs | **9** | horizon | **18** | portion tube opening |
| **1** | apxs cal target | **10** | inlet | **19** | rems uv sensor |
| **2** | chemcam cal target | **11** | mahli | **20** | rover rear deck |
| **3** | chemin inlet open | **12** | mahli cal target | **21** | scoop |
| **4** | drill | **13** | mastcam | **22** | sun |
| **5** | drill holes | **14** | mastcam cal target | **23** | turret |
| **6** | drt front | **15** | observation tray | **24** | wheel |
| **7** | drt side | **16** | portion box | | |
| **8** | ground | **17** | portion tube | | |


**Notes**: Sun (22) has no examples anywhere, several classes are absent from validation or test, and class proportions differ substantially between splits.


For this reason, new stratified splits will be created. 70% will be used to train, and the remaining 30% is split evenly between validation and test sets. The combined data from all 3 CSV's will be used for this, so that the labels stay attached to their respective images.

In [ ]:
# keep original assignment for comparison
original_df = df.copy()

# 70% training
new_train, held_out = train_test_split(
    original_df,
    test_size=0.30,
    stratify=original_df["label"],
    random_state=42
)

# 15% validation, 15% test
new_validation, new_test = train_test_split(
    held_out,
    test_size=0.50,
    stratify=held_out["label"],
    random_state=42
)

# record new assignments without modifying original CSVs
new_train = new_train.assign(split="train")
new_validation = new_validation.assign(split="validation")
new_test = new_test.assign(split="test")

df = pd.concat(
    [new_train, new_validation, new_test],
    ignore_index=True
)

display(df.head(10))  # show the first ten rows
display(df["split"].value_counts())  # count rows in each split

Comparison of label percentages in train, validation, and data splits.

In [ ]:
id_to_name = {}  # maps numeric ID to class name
with (DATA_DIR / "Labels Information.txt").open(encoding="utf-8-sig") as handle:
    for line in handle:
        if line.strip():
            label_id, class_name = line.strip().split(maxsplit=1)
            id_to_name[int(label_id)] = class_name

def summarize_labels(data):
    # count each label in each set
    counts = pd.crosstab(data["label"], data["split"])

    # include all defined labels, even those with 0 images
    counts = counts.reindex(
        index=sorted(id_to_name),
        columns=["train", "validation", "test"],
        fill_value=0
    )

    # summary table with class names
    summary = pd.DataFrame(index=counts.index)
    summary["class_name"] = summary.index.map(id_to_name)

    # add count and percentage for each set
    for split in ["train", "validation", "test"]:
        summary[f"{split}_count"] = counts[split]

        summary[f"{split}_percent"] = (
            100 * counts[split] / counts[split].sum()
        ).round(2)

    return summary.rename_axis("label").reset_index()


# create both summaries (ogirinal & resplit)
original_summary = summarize_labels(original_df)
new_summary = summarize_labels(df)

# display both tables
print("ORIGINAL SPLITS — percentages within each set")
display(original_summary)
print("NEW STRATIFIED SPLITS — percentages within each set")
display(new_summary)

# combine both summaries into one CSV
original_summary.insert(0, "split_strategy", "original")
new_summary.insert(0, "split_strategy", "stratified")
comparison = pd.concat(
    [original_summary, new_summary],
    ignore_index=True
)

OUT_DIR.mkdir(exist_ok=True)
output_path = OUT_DIR / "original_vs_stratified_class_percentages.csv"
comparison.to_csv(output_path, index=False)

print("Saved to:", output_path)